In [ ]:
import os
import sys

import pandas as pd

sys.path.append(os.path.abspath("../"))

In [ ]:
def load_and_process_runs(base_path, prefix):
    runs = [f for f in os.listdir(base_path) if f.startswith(prefix) and f.endswith(".csv")]
    dfs = [pd.read_csv(os.path.join(base_path, f)) for f in runs]
    return dfs


grayscale_path = "../../data/evaluation/cpn-grayscale"
yuyv_path = "../../data/evaluation/cpn-yuyv"

grayscale_runs = load_and_process_runs(grayscale_path, "run_")
yuyv_runs = load_and_process_runs(yuyv_path, "run_")
id_cols = [
    col for col in grayscale_runs[0].columns if col.startswith(("resolution", "architecture"))
]

print(id_cols)

diffs = []
for i in range(min(len(grayscale_runs), len(yuyv_runs))):
    df_grayscale = grayscale_runs[i].copy()
    df_yuyv = yuyv_runs[i]
for col in df_grayscale.columns:
    if col.startswith(("classifier", "encoder")) and "architecture" not in col:
        df_grayscale[col] = df_grayscale[col] - df_yuyv[col]
    diffs.append(df_grayscale)

mean_grayscale = pd.concat(grayscale_runs).groupby(id_cols).mean(numeric_only=True).reset_index()
std_grayscale = pd.concat(grayscale_runs).groupby(id_cols).std(numeric_only=True).reset_index()
mean_yuyv = pd.concat(yuyv_runs).groupby(id_cols).mean(numeric_only=True).reset_index()
std_yuyv = pd.concat(yuyv_runs).groupby(id_cols).std(numeric_only=True).reset_index()

mean_diff = pd.concat(diffs).groupby(id_cols).mean(numeric_only=True).reset_index()
std_diff = pd.concat(diffs).groupby(id_cols).std(numeric_only=True).reset_index()

mean_grayscale.to_csv(f"{grayscale_path}/mean.csv", index=False)
std_grayscale.to_csv(f"{grayscale_path}/std.csv", index=False)
mean_yuyv.to_csv(f"{yuyv_path}/mean.csv", index=False)
std_yuyv.to_csv(f"{yuyv_path}/std.csv", index=False)
mean_diff.to_csv("../../data/evaluation/cpn-grayscale-diff-mean.csv", index=False)
std_diff.to_csv("../../data/evaluation/cpn-grayscale-diff-std.csv", index=False)


### Build Latex Table


In [ ]:
def arch_key(arch):
    parts = arch.split("_")
    return f"{parts[0]}_{parts[-1]}"
# e.g. "ires_16x16_v1" → "ires_v1", "conv_32x32_v1" → "conv_v1"

arch_mapping = {
    "ires_v1": "CPN$_1$",
    "ires_v2": "CPN$_2$",
    "conv_v1": "CPN$_3$",
}

# mean_df = mean_yuyv
# std_df = std_yuyv

mean_df = mean_grayscale
std_df = std_grayscale

# mean_df = mean_diff
# std_df = std_diff

rounding_precision_recall = 1
rounding_precision_euc = 1
rounding_std = 2

table = (
    mean_df.groupby(["resolution", "architecture"])
    .apply(
        lambda x: f"& {arch_mapping[arch_key(x['architecture'].iloc[0])]} & "
        + " & ".join(
            [
                f"${round(x['encoder_recall_at_k_ball'].iloc[0] * 100, rounding_precision_recall):.{rounding_precision_recall}f} \pm {round(std_df[(std_df['resolution'] == x['resolution'].iloc[0]) & (std_df['architecture'] == x['architecture'].iloc[0])]['encoder_recall_at_k_ball'].iloc[0] * 100, rounding_std):.{rounding_std}f}$",
                
                f"${round(x['encoder_recall_at_k_penaltyMark'].iloc[0] * 100, rounding_precision_recall):.{rounding_precision_recall}f} \pm {round(std_df[(std_df['resolution'] == x['resolution'].iloc[0]) & (std_df['architecture'] == x['architecture'].iloc[0])]['encoder_recall_at_k_penaltyMark'].iloc[0] * 100, rounding_std):.{rounding_std}f}$",
                
                f"${round(x['encoder_recall_at_k_intersections'].iloc[0] * 100, rounding_precision_recall):.{rounding_precision_recall}f} \pm {round(std_df[(std_df['resolution'] == x['resolution'].iloc[0]) & (std_df['architecture'] == x['architecture'].iloc[0])]['encoder_recall_at_k_intersections'].iloc[0] * 100, rounding_std):.{rounding_std}f}$",
                
                f"${round(x['encoder_mae_ball'].iloc[0], rounding_precision_euc):.{rounding_precision_euc}f} \pm {round(std_df[(std_df['resolution'] == x['resolution'].iloc[0]) & (std_df['architecture'] == x['architecture'].iloc[0])]['encoder_mae_ball'].iloc[0], rounding_std):.{rounding_std}f}$",
                
                f"${round(x['encoder_mae_penaltyMark'].iloc[0], rounding_precision_euc):.{rounding_precision_euc}f} \pm {round(std_df[(std_df['resolution'] == x['resolution'].iloc[0]) & (std_df['architecture'] == x['architecture'].iloc[0])]['encoder_mae_penaltyMark'].iloc[0], rounding_std):.{rounding_std}f}$",
                
                f"${round(x['encoder_mae_intersections'].iloc[0], rounding_precision_euc):.{rounding_precision_euc}f} \pm {round(std_df[(std_df['resolution'] == x['resolution'].iloc[0]) & (std_df['architecture'] == x['architecture'].iloc[0])]['encoder_mae_intersections'].iloc[0], rounding_std):.{rounding_std}f}$",
            ]
        )
        + " \\\\"
    )
    .reset_index()
)
table["arch_order"] = table["architecture"].apply(lambda a: arch_mapping[arch_key(a)])
diff_table = table.sort_values(["resolution", "arch_order"]).drop(columns="arch_order")
diff_table.columns = ["resolution", "architecture", "formatted_string"]

for resolution in sorted(diff_table["resolution"].unique(), reverse=True):
    print(resolution)
    print(
        "\n".join(diff_table[diff_table["resolution"] == resolution]["formatted_string"].tolist())
    )